# Homework 1

#### EE-556 Mathematics of Data - Fall 2026


In this homework, we consider a multiclass classification task modeled by multinomial (softmax) logistic regression. Your goal will be to analyze the estimator and its properties (convexity, existence/uniqueness), and to derive gradients/Hessians and smoothness bounds. The first part consists of theoretical questions only.


<div class="alert alert-info">
  ℹ️ <strong>Information on group based work:</strong>
</div>

- Only one group member needs to submit all five completed notebooks on behalf of the group.
- You may seek help outside your group, but each group must write its own answers.
- If the work was <span style="color: red;">unevenly distributed</span> among group members, let the TAs know.
- Only one member of the group is allowed to use AI. We will require sharing the conversation history with the AI in the form of a public link. If you use multiple conversations across the same or multiple tools please share all of them. Name the person in your group who is allowed to use AI. We encourage you to use the AI to help you understand the material, but we ask you to write the code and theory solutions by yourself. Part 5 permits AI usage as specified in its instructions.

<div style="border: 1px solid #f00; background-color: #fdd; padding: 10px; border-radius: 5px;">
  ⚠️ Do not forget: List each group member's name and SCIPER number.
</div>


Person 1 **Name**: Enric Guasch Mesià || Person 1 **SCIPER**: 415498


Person 2 **Name**: || Person 2 **SCIPER**:


Person 3 **Name**: || Person 3 **SCIPER**:


<div style="border: 1px solid #0a0; background-color: #dfd; padding: 10px; border-radius: 5px;">
  📓 Feedback on AI use: Please use the following cell to provide feedback on the AI use in this notebook.
  
  For example, how useful were the tools to you? Which tools did you use? Did you feel like they helped you understand the material better?
</div>

*Describe your use of AI tools here.*


## 1. Multiclass Softmax Logistic Regression - 30 Points


We now model multiclass classification with classes $c \in \{1,\dots,C\}$. For each sample $(\mathbf{a}_i, b_i)$ with $\mathbf{a}_i \in \mathbb{R}^p$ and $b_i \in \{1,\dots,C\}$, let $\mathbf{X} = [\mathbf{x}_1,\dots,\mathbf{x}_C] \in \mathbb{R}^{p\times C}$ be the class weight matrix. The softmax model defines

$$
\mathbb{P}(b_i = c \mid \mathbf{a}_i) = \frac{\exp(\mathbf{a}_i^\top \mathbf{x}_c)}{\sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k)}.
$$

Assume i.i.d. samples $\{(\mathbf{a}_i,b_i)\}_{i=1}^n$. Our goal is to estimate $\mathbf{X}$ by maximum likelihood (and later with an $\ell_2$ regularizer).


__(a)__ (2 points) Show that the negative log-likelihood $f$ can be written as:

$$
\begin{aligned}
 f(\mathbf{X})
 &= - \log \mathbb{P}(b_1,\dots,b_n\mid \mathbf{a}_1,\dots,\mathbf{a}_n)\\
 &= \sum_{i=1}^n \left[ -\mathbf{a}_i^\top \mathbf{x}_{b_i} + \log \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \right].
\end{aligned}
$$



*Write your answer here.*

The samples are independent and identically distributed (i.i.d.), so the joint probability will be the product of the individual probabilities. If we take the negative logarithm it converts this product into a sum:

$$\begin{aligned} f(\mathbf{X}) &= - \log \mathbb{P}(b_1,\dots,b_n\mid \mathbf{a}_1,\dots,\mathbf{a}_n) \\ &= - \log \prod_{i=1}^n \mathbb{P}(b_i \mid \mathbf{a}_i) \\ &= - \sum_{i=1}^n \log \mathbb{P}(b_i \mid \mathbf{a}_i) \end{aligned}$$

We substitute the definition of the softmax probability into the equation:

$$\begin{aligned} f(\mathbf{X}) &= - \sum_{i=1}^n \log \left( \frac{\exp(\mathbf{a}_i^\top \mathbf{x}_{b_i})}{\sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k)} \right) \end{aligned}$$

Using logarithm properties $\log(A/B) = \log(A) - \log(B)$ and also $\log(\exp(x)) = x$, this simplifies to:

$$\begin{aligned} f(\mathbf{X}) &= - \sum_{i=1}^n \left( \log(\exp(\mathbf{a}_i^\top \mathbf{x}_{b_i})) - \log \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \right) \\ &= - \sum_{i=1}^n \left( \mathbf{a}_i^\top \mathbf{x}_{b_i} - \log \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \right) \\ &= \sum_{i=1}^n \left[ -\mathbf{a}_i^\top \mathbf{x}_{b_i} + \log \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \right] \end{aligned}$$

__(b)__ (3 points) Show that $\mathbf{u} \mapsto \log\!\left(\sum_{k=1}^C e^{u_k}\right)$ is convex on $\mathbb{R}^C$. Then, show that $f(\mathbf{X})$ is convex.


Hint: use Jensen's inequality.

*Write your answer here.*

To show that $g(\mathbf{u}) = \log\left(\sum_{k=1}^C e^{u_k}\right)$ is convex, we can analyze its Hessian. Let $p_k = \frac{e^{u_k}}{\sum_{j=1}^C e^{u_j}}$, which forms a valid probability distribution since $p_k > 0$ and $\sum_{k=1}^C p_k = 1$.

The first derivative with respect to $u_k$ is:


$$\frac{\partial g}{\partial u_k} = p_k$$

The second derivative (Hessian matrix $\mathbf{H} = \nabla^2 g(\mathbf{u})$) is:


$$H_{kl} = \frac{\partial p_k}{\partial u_l} = \begin{cases} p_k (1 - p_k) & \text{if } k = l \\ -p_k p_l & \text{if } k \neq l \end{cases}$$

This means $\mathbf{H} = \operatorname{Diag}(\mathbf{p}) - \mathbf{p}\mathbf{p}^\top$. To prove $\mathbf{H}$ is positive semi-definite (PSD), we take an arbitrary vector $\mathbf{z} \in \mathbb{R}^C$ and evaluate $\mathbf{z}^\top \mathbf{H} \mathbf{z}$:


$$\begin{aligned} \mathbf{z}^\top \mathbf{H} \mathbf{z} &= \sum_{k=1}^C p_k z_k^2 - \sum_{k=1}^C \sum_{l=1}^C p_k p_l z_k z_l \\ &= \sum_{k=1}^C p_k z_k^2 - \left( \sum_{k=1}^C p_k z_k \right)^2 \end{aligned}$$

Let $Z$ be a random variable taking value $z_k$ with probability $p_k$. By Jensen's inequality (or the property of variance where $\operatorname{Var}(Z) = \mathbb{E}[Z^2] - (\mathbb{E}[Z])^2 \ge 0$), we have:


$$\mathbf{z}^\top \mathbf{H} \mathbf{z} = \mathbb{E}[Z^2] - (\mathbb{E}[Z])^2 \ge 0$$


Since the Hessian is positive semi-definite, $g(\mathbf{u})$ is a convex function.

Now, we evaluate the full function $f(\mathbf{X})$:


$$f(\mathbf{X}) = \sum_{i=1}^n \left[ -\mathbf{a}_i^\top \mathbf{x}_{b_i} + \log \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \right]$$

1. The first term inside the sum, $-\mathbf{a}_i^\top \mathbf{x}_{b_i}$, is a linear function of $\mathbf{X}$ and is therefore convex.
2. The second term is the composition of the log-sum-exp function $g(\mathbf{u})$ with an affine mapping $\mathbf{u}_i = \mathbf{X}^\top \mathbf{a}_i$. The composition of a convex function with an affine mapping preserves convexity.
3. Because the sum of convex functions is convex, the overall negative log-likelihood function $f(\mathbf{X})$ is convex.


Since the negative log-likelihood $f$ is convex, every local minimizer is a global minimizer. The maximum likelihood estimation problem is
$$
\mathbf{X}^\star_{ML} = \arg\min_{\mathbf{X} \in \mathbb{R}^{p\times C}} f(\mathbf{X})
$$

But does $f$ always attain its infimum? The following three questions examine this issue.


__(c)__ (1 point) Explain the difference between infima and minima. Give an example of a convex function on $\mathbb{R}$ that does not attain its infimum.


*Write your answer here.*

The infimum of $f$ over a set $\mathcal{D}$ is the greatest lower bound of the values $\{f(x) : x\in\mathcal{D}\}$. It always exists in the extended reals (possibly $-\infty$), but nothing guarantees us that some point actually achieves it. A minimum is an infimum that *is* achieved: $f$ attains its minimum if there exists $x^\star\in\mathcal{D}$ with $f(x^\star) = \inf_{x\in\mathcal{D}} f(x)$. So every minimum is an infimum, but not the other way around.

A simple convex example is $f(x) = e^x$ on $\mathbb{R}$. It is convex since $f''(x) = e^x > 0$, and $\inf_x e^x = 0$ because $e^x\to 0$ as $x\to-\infty$. However, $e^x > 0$ for every real $x$, so the value $0$ is never reached and $f$ has no minimizer. 

The same happens with $x\mapsto\log(1+e^{-x})$, which is exactly the binary logistic loss and already gives us a hint of what goes wrong in (e).

__(d)__ (2 points) Assume there exists $\mathbf{X}_0 \in \mathbb{R}^{p\times C}$ such that for all $i$,
$$
\mathbf{a}_i^\top \mathbf{x}_{0, b_i} - \max_{k \neq b_i} \mathbf{a}_i^\top \mathbf{x}_{0,k} > 0.
$$
This is called one-versus-all complete separation in multiclass settings. Give a geometric interpretation (e.g., for $p=2$) and explain why the name is appropriate.


*Write your answer here.*

Let's first interpret the condition. The classifier induced by $\mathbf{X}_0$ predicts the class with the largest score, $\hat b(\mathbf{a}) = \arg\max_k \mathbf{a}^\top\mathbf{x}_{0,k}$. The assumption says that, for every sample, the score of its true class is *strictly* larger than the score of every other class, that is every training point is correctly classified with a positive margin.

If we look at it geometrically, $\mathbf{X}_0$ splits the feature space into the decision regions
$$
\mathcal{R}_c = \{\mathbf{a}\in\mathbb{R}^p : \mathbf{a}^\top\mathbf{x}_{0,c} > \mathbf{a}^\top\mathbf{x}_{0,k}\ \ \forall k\neq c\} = \bigcap_{k\neq c}\{\mathbf{a} : \mathbf{a}^\top(\mathbf{x}_{0,c}-\mathbf{x}_{0,k}) > 0\},
$$
each one is an intersection of open half-spaces that are bounded by the hyperplanes $\mathbf{a}^\top(\mathbf{x}_{0,c}-\mathbf{x}_{0,k}) = 0$. Since the model has no intercept, these hyperplanes pass through the origin, so every $\mathcal{R}_c$ is a convex polyhedral cone. For $p=2$ it's very straightforward: lines through the origin cut the plane into (at most) $C$ wedge-shaped sectors, like slices of a pizza, one per class, and complete separation means that every point of class $c$ lies strictly inside the sector of class $c$, none on a boundary ray. For $C=2$ this reduces to the typical binary picture of a single line through the origin with the two classes on opposite sides.

Why the name: fix a class $c$. Points of class $c$ satisfy $\mathbf{a}_i^\top\mathbf{x}_{0,c} > \mathbf{a}_i^\top\mathbf{x}_{0,k}$ for all $k\neq c$, so they lie in $\mathcal{R}_c$, while a point of any other class $k$ has $\mathbf{a}_i^\top\mathbf{x}_{0,k} > \mathbf{a}_i^\top\mathbf{x}_{0,c}$ and therefore lies outside $\mathcal{R}_c$. So the region $\mathcal{R}_c$ separates class $c$ from the union of *all* the other classes, which is the "one-versus-all" part, and this holds for every class simultaneously with the same $\mathbf{X}_0$. It is "complete" because the separation is perfect and strict, as no training sample is misclassified or lies on a decision boundary.

Complete separation allows perfect classification of the training data. However, as you will show next, it prevents the existence of a finite maximum likelihood estimate.


__(e)__ (3 points) In a one-versus-all complete separation setting (as in (d)), prove that $f$ does not attain its minimum. Hint: consider $f(\alpha \mathbf{X}_0)$ as $\alpha \to +\infty$ and compare it to $f(\mathbf{X}_0)$.


*Write your answer here.*

Our intuition is that under complete separation we can always make the model "more confident" by scaling up $\mathbf{X}_0$: all predictions stay correct while every probability $\mathbb{P}(b_i\mid\mathbf{a}_i)$ gets pushed closer to $1$, so the loss keeps decreasing and the optimum tends to infinity.

First, we rewrite each term of $f$ by factoring out the score of the true class:
$$
-\mathbf{a}_i^\top\mathbf{x}_{b_i} + \log\sum_{k=1}^C e^{\mathbf{a}_i^\top\mathbf{x}_k} = \log\Big(1 + \sum_{k\neq b_i} e^{-\mathbf{a}_i^\top(\mathbf{x}_{b_i}-\mathbf{x}_k)}\Big).
$$

**$f$ is strictly positive.** The sum inside the logarithm is strictly positive (since $C\ge 2$), so every term is $>0$ and $f(\mathbf{X})>0$ for all $\mathbf{X}$. In particular $\inf f\ge 0$.

**Scaling $\mathbf{X}_0$ drives $f$ to $0$.** Let $d_{ik} = \mathbf{a}_i^\top(\mathbf{x}_{0,b_i}-\mathbf{x}_{0,k})$ for $k\neq b_i$. By assumption, $d_{ik}\ge m_i := \mathbf{a}_i^\top\mathbf{x}_{0,b_i} - \max_{k\neq b_i}\mathbf{a}_i^\top\mathbf{x}_{0,k} > 0$. For $\alpha>0$,
$$
f(\alpha\mathbf{X}_0) = \sum_{i=1}^n\log\Big(1+\sum_{k\neq b_i}e^{-\alpha d_{ik}}\Big) \le \sum_{i=1}^n\log\big(1+(C-1)\,e^{-\alpha m_i}\big) \;\longrightarrow\; 0 \quad \text{as } \alpha\to+\infty.
$$
Moreover, since all $d_{ik}>0$, each $e^{-\alpha d_{ik}}$ is strictly decreasing in $\alpha$, so $\alpha\mapsto f(\alpha\mathbf{X}_0)$ is strictly decreasing. In particular $f(\alpha\mathbf{X}_0) < f(\mathbf{X}_0)$ for every $\alpha>1$: a scaled-up version of the separating solution is always strictly better than the solution itself.

**Conclusion.** The two steps give $\inf_{\mathbf{X}} f(\mathbf{X}) = 0$. Let us suppose some $\mathbf{X}^\star$ attained the minimum; then $f(\mathbf{X}^\star) = 0$, which contradicts the strict positivity of $f$ that we just proved. Said differently, for any candidate $\mathbf{X}^\star$ we have $f(\mathbf{X}^\star)>0$, and we can pick $\alpha$ large enough so that $f(\alpha\mathbf{X}_0) < f(\mathbf{X}^\star)$. So $f$ does not attain its minimum: the infimum is only approached along $\|\alpha\mathbf{X}_0\|_F\to\infty$, and no finite maximum likelihood estimate exists.

We resolve this issue by adding a regularizer. Consider the regularized function

$$
 f_\mu(\mathbf{X}) = f(\mathbf{X}) + \frac{\mu}{2} \|\mathbf{X}\|_F^2, \quad \mu > 0.
$$

__(f)__ (2 points) Show that the gradient with respect to $\mathbf{X}$ of $f_\mu$ can be expressed as
$$
 \nabla_{\mathbf{X}} f_\mu(\mathbf{X}) = \sum_{i=1}^n \mathbf{a}_i \big( \mathbf{p}_i - \mathbf{e}_{b_i} \big)^\top + \mu \mathbf{X},
$$
where $\mathbf{e}_{b_i} \in \mathbb{R}^C$ is the [one-hot vector](https://en.wikipedia.org/wiki/One-hot) for class $b_i$, $\mathbf{p}_i \in \mathbb{R}^C$ has entries $p_{i,c} = \mathbb{P}(b_i=c\mid \mathbf{a}_i)$ under the softmax model, and $\mathbf{a}_i(\mathbf{p}_i - \mathbf{e}_{b_i})^\top\in\mathbb{R}^{p\times C}$ denotes the outer product.



*Write your answer here.*

The cleanest way we can do this is to differentiate with respect to one column $\mathbf{x}_c$ at a time and then stack the columns back into a matrix. Fix a sample $i$ and a class $c$ and look at the two parts of the $i$-th summand of $f$.

The linear part $-\mathbf{a}_i^\top\mathbf{x}_{b_i}$ only involves the column $\mathbf{x}_{b_i}$, so its gradient with respect to $\mathbf{x}_c$ is $-\mathbf{a}_i$ if $c=b_i$ and $\mathbf{0}$ otherwise, so we can write it as $-[\mathbf{e}_{b_i}]_c\,\mathbf{a}_i$. For the log-sum-exp part, the chain rule gives
$$
\nabla_{\mathbf{x}_c}\log\sum_{k=1}^C e^{\mathbf{a}_i^\top\mathbf{x}_k} = \frac{e^{\mathbf{a}_i^\top\mathbf{x}_c}}{\sum_{k=1}^C e^{\mathbf{a}_i^\top\mathbf{x}_k}}\,\mathbf{a}_i = p_{i,c}\,\mathbf{a}_i.
$$
Finally, the regularizer is $\frac{\mu}{2}\|\mathbf{X}\|_F^2 = \frac{\mu}{2}\sum_c\|\mathbf{x}_c\|_2^2$, whose gradient with respect to $\mathbf{x}_c$ is $\mu\mathbf{x}_c$. Putting everything together,
$$
\nabla_{\mathbf{x}_c}f_\mu(\mathbf{X}) = \sum_{i=1}^n \mathbf{a}_i\big(p_{i,c}-[\mathbf{e}_{b_i}]_c\big) + \mu\,\mathbf{x}_c.
$$

The gradient with respect to $\mathbf{X}$ is the $p\times C$ matrix whose $c$-th column is $\nabla_{\mathbf{x}_c}f_\mu(\mathbf{X})$. The matrix whose $c$-th column is $\mathbf{a}_i(p_{i,c}-[\mathbf{e}_{b_i}]_c)$ is exactly the outer product $\mathbf{a}_i(\mathbf{p}_i-\mathbf{e}_{b_i})^\top$, and stacking the columns $\mu\mathbf{x}_c$ gives $\mu\mathbf{X}$. Therefore
$$
\nabla_{\mathbf{X}}f_\mu(\mathbf{X}) = \sum_{i=1}^n\mathbf{a}_i(\mathbf{p}_i-\mathbf{e}_{b_i})^\top + \mu\mathbf{X}.
$$
This means each sample contributes its feature vector weighted by the "prediction error" $\mathbf{p}_i-\mathbf{e}_{b_i}$ between the predicted and the observed class distribution.

__(g)__ (4 points) Use row-wise vectorization of $\mathbf{X}$, obtained by stacking its rows one after another into a vector in $\mathbb{R}^{pC}$. Show that the Hessian of $f_\mu$ in these coordinates can be written as
<a id="eq-softmax-hessian"></a>

$$
 \nabla^2 f_\mu(\mathbf{X}) = \sum_{i=1}^n (\mathbf{a}_i\mathbf{a}_i^\top) \otimes \big( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top \big) + \mu \mathbf{I}_{pC},\tag{1}
$$
where $\otimes$ is the Kronecker product, and $\operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top$ is the softmax Jacobian, which is positive semidefinite.


*Write your answer here.*

The Hessian is just the matrix of second-order partial derivatives, so my plan is to compute $\frac{\partial^2 f_\mu}{\partial X_{jc}\,\partial X_{lk}}$ entrywise and then recognize the Kronecker structure.

From (f), the $(j,c)$ entry of the gradient is
$$
\frac{\partial f_\mu}{\partial X_{jc}} = \sum_{i=1}^n a_{ij}\big(p_{i,c}-[\mathbf{e}_{b_i}]_c\big) + \mu X_{jc}.
$$
Apart from the regularizer, only $p_{i,c}$ depends on $\mathbf{X}$, and it depends through the scores $\mathbf{u}_i = \mathbf{X}^\top\mathbf{a}_i$, i.e. $u_{i,k} = \sum_l a_{il}X_{lk}$. So $X_{lk}$ only affects the $k$-th score, with $\partial u_{i,k}/\partial X_{lk} = a_{il}$. Using the softmax Jacobian from (b), $\partial p_{i,c}/\partial u_{i,k} = p_{i,c}(\delta_{ck}-p_{i,k})$, the chain rule gives $\partial p_{i,c}/\partial X_{lk} = a_{il}\,p_{i,c}(\delta_{ck}-p_{i,k})$, and thus
$$
\frac{\partial^2 f_\mu}{\partial X_{jc}\,\partial X_{lk}} = \sum_{i=1}^n a_{ij}a_{il}\big(\delta_{ck}\,p_{i,c}-p_{i,c}\,p_{i,k}\big) + \mu\,\delta_{jl}\delta_{ck} = \sum_{i=1}^n(\mathbf{a}_i\mathbf{a}_i^\top)_{jl}\big(\operatorname{Diag}(\mathbf{p}_i)-\mathbf{p}_i\mathbf{p}_i^\top\big)_{ck} + \mu\,\delta_{jl}\delta_{ck}.
$$

Now the vectorization matters. With row-wise stacking, $X_{jc}$ sits at position $(j-1)C+c$: the row index $j$ selects a block of size $C$, and the column index $c$ is the position inside that block. On the other hand, for $\mathbf{M}\in\mathbb{R}^{p\times p}$ and $\mathbf{N}\in\mathbb{R}^{C\times C}$, the Kronecker product $\mathbf{M}\otimes\mathbf{N}$ is the $p\times p$ block matrix with blocks $M_{jl}\mathbf{N}$, so its entry at position $\big((j-1)C+c,\,(l-1)C+k\big)$ is exactly $M_{jl}N_{ck}$. This is exactly the structure above with $\mathbf{M}=\mathbf{a}_i\mathbf{a}_i^\top$ and $\mathbf{N}=\operatorname{Diag}(\mathbf{p}_i)-\mathbf{p}_i\mathbf{p}_i^\top$, while $\delta_{jl}\delta_{ck}$ are the entries of $\mathbf{I}_p\otimes\mathbf{I}_C = \mathbf{I}_{pC}$. Therefore
$$
\nabla^2 f_\mu(\mathbf{X}) = \sum_{i=1}^n(\mathbf{a}_i\mathbf{a}_i^\top)\otimes\big(\operatorname{Diag}(\mathbf{p}_i)-\mathbf{p}_i\mathbf{p}_i^\top\big) + \mu\mathbf{I}_{pC}.
$$


__(h)__ (2 points) Show that $f_\mu$ is $\mu$-strongly convex.


*Write your answer here.*

A twice-differentiable function is $\mu$-strongly convex if and only if its Hessian satisfies $\nabla^2 f_\mu(\mathbf{X})\succeq\mu\mathbf{I}$ everywhere. Looking at (1), it is enough to show that the data term is positive semidefinite.

For each $i$, both factors of the Kronecker product are PSD: $\mathbf{a}_i\mathbf{a}_i^\top$ because $\mathbf{v}^\top\mathbf{a}_i\mathbf{a}_i^\top\mathbf{v} = (\mathbf{a}_i^\top\mathbf{v})^2\ge 0$, and $\operatorname{Diag}(\mathbf{p}_i)-\mathbf{p}_i\mathbf{p}_i^\top$ by the variance argument in (b). The eigenvalues of the Kronecker product of two symmetric matrices are all the pairwise products $\lambda_j(\mathbf{M})\lambda_k(\mathbf{N})$ of their eigenvalues, which here are all nonnegative. So each term $(\mathbf{a}_i\mathbf{a}_i^\top)\otimes(\operatorname{Diag}(\mathbf{p}_i)-\mathbf{p}_i\mathbf{p}_i^\top)$ is PSD, and so is their sum. Therefore, for every $\mathbf{X}$,
$$
\nabla^2 f_\mu(\mathbf{X}) = \underbrace{\sum_{i=1}^n(\mathbf{a}_i\mathbf{a}_i^\top)\otimes\big(\operatorname{Diag}(\mathbf{p}_i)-\mathbf{p}_i\mathbf{p}_i^\top\big)}_{\succeq\,0} + \mu\mathbf{I}_{pC}\;\succeq\;\mu\mathbf{I}_{pC},
$$
and $f_\mu$ is $\mu$-strongly convex.

__(i)__ (2 points) Is it possible for a strongly convex function to not attain its minimum? Justify your reasoning (you may assume the domain is $\mathbb{R}^{p\times C}$).


*Write your answer here.*

No. On the whole space $\mathbb{R}^{p\times C}$, a (finite-valued) strongly convex function always attains its minimum, and the minimizer is unique. The reason is that strong convexity forces at least quadratic growth, which rules out the "escape to infinity" we saw in (e).

If we take $f_\mu$ and fix any point $\mathbf{X}_0$, strong convexity and Cauchy–Schwarz give, for all $\mathbf{Y}$,
$$
f_\mu(\mathbf{Y}) \ge f_\mu(\mathbf{X}_0) + \langle\nabla f_\mu(\mathbf{X}_0),\mathbf{Y}-\mathbf{X}_0\rangle + \frac{\mu}{2}\|\mathbf{Y}-\mathbf{X}_0\|_F^2 \ge f_\mu(\mathbf{X}_0) - \|\nabla f_\mu(\mathbf{X}_0)\|_F\,\|\mathbf{Y}-\mathbf{X}_0\|_F + \frac{\mu}{2}\|\mathbf{Y}-\mathbf{X}_0\|_F^2.
$$
The right-hand side grows quadratically, so $f_\mu(\mathbf{Y})\to+\infty$ as $\|\mathbf{Y}\|_F\to\infty$. So the sublevel set $\mathcal{S} = \{\mathbf{Y} : f_\mu(\mathbf{Y})\le f_\mu(\mathbf{X}_0)\}$ is bounded and it is also closed because $f_\mu$ is continuous, and nonempty since it contains $\mathbf{X}_0$. By Weierstrass' theorem $f_\mu$ attains its minimum on the compact set $\mathcal{S}$, and since every point outside $\mathcal{S}$ has a value larger than $f_\mu(\mathbf{X}_0)$, this is a global minimizer. Uniqueness comes from strict convexity: if $\mathbf{X}_1\neq\mathbf{X}_2$ were both minimizers, their midpoint would have a strictly smaller value.

The assumption on the domain does matter: on a non-closed domain a strongly convex function can miss its infimum, for example $x\mapsto x^2$ on the open interval $(1,2)$ has infimum $1$, which is never attained. On $\mathbb{R}^{p\times C}$ this cannot happen, which is exactly why the regularized problem (2) has a well-defined and unique solution $\mathbf{X}^\star$, even in the separable case where the unregularized problem has none.

We will now show that $f_\mu$ is smooth, i.e., $\nabla f_\mu$ is L-Lipschitz with respect to the Frobenius norm, with a simple conservative bound
$$
 L = \|\mathbf{A}\|_F^2 + \mu.
$$
where
$$
 \mathbf{A} = \begin{bmatrix}
  \leftarrow &  \mathbf{a}_1^\top & \rightarrow \\
  \leftarrow &  \mathbf{a}_2^\top & \rightarrow \\
   &  \ldots &  \\
  \leftarrow &  \mathbf{a}_n^\top & \rightarrow \\
 \end{bmatrix}.
$$
(You may use that the operator norm of the softmax Jacobian is bounded by $1/2$, and a looser bound $\le 1$ is acceptable for grading.)

Hint: check the properties of the spectral norm with respect to dot product, Kronecker product, and outer product.

(4 points for all three questions)


__(j-1)__ Show that $\lambda_{\max}(\mathbf{a}_i\mathbf{a}_i^T) = \left\| \mathbf{a}_i\right\|_2^2$, where $\lambda_{\max}(\cdot)$ denotes the largest eigenvalue.


*Write your answer here.*

The matrix $\mathbf{a}_i\mathbf{a}_i^\top$ is symmetric with rank at most one, so the idea is to find its eigenvalues explicitly. If $\mathbf{a}_i=\mathbf{0}$ the matrix is zero and the claim is trivial, so we assume $\mathbf{a}_i\neq\mathbf{0}$.

Taking $\mathbf{v}=\mathbf{a}_i$ gives $(\mathbf{a}_i\mathbf{a}_i^\top)\mathbf{a}_i = \mathbf{a}_i(\mathbf{a}_i^\top\mathbf{a}_i) = \|\mathbf{a}_i\|_2^2\,\mathbf{a}_i$, so $\|\mathbf{a}_i\|_2^2$ is an eigenvalue. For any $\mathbf{v}\perp\mathbf{a}_i$ we get $(\mathbf{a}_i\mathbf{a}_i^\top)\mathbf{v} = \mathbf{a}_i(\mathbf{a}_i^\top\mathbf{v}) = \mathbf{0}$, so $0$ is an eigenvalue with an eigenspace of dimension $p-1$. This accounts for all $p$ eigenvalues (consistent with $\operatorname{tr}(\mathbf{a}_i\mathbf{a}_i^\top) = \|\mathbf{a}_i\|_2^2$), so
$$
\lambda_{\max}(\mathbf{a}_i\mathbf{a}_i^\top) = \|\mathbf{a}_i\|_2^2.
$$
Alternatively, via the Rayleigh quotient: $\lambda_{\max}(\mathbf{a}_i\mathbf{a}_i^\top) = \max_{\|\mathbf{v}\|_2=1}(\mathbf{a}_i^\top\mathbf{v})^2\le\|\mathbf{a}_i\|_2^2$ by Cauchy–Schwarz, with equality at $\mathbf{v}=\mathbf{a}_i/\|\mathbf{a}_i\|_2$.

__(j-2)__ Using [(1)](#eq-softmax-hessian), show that $\lambda_{\max}(\nabla^2 f_\mu(\mathbf{X})) \leq \sum_{i=1}^{n} \|\mathbf{a}_i\|_2^2 + \mu$.

*Write your answer here.*

We write $\mathbf{S}_i = \operatorname{Diag}(\mathbf{p}_i)-\mathbf{p}_i\mathbf{p}_i^\top$. Three properties of symmetric matrices help us:

1. **Sum:** $\lambda_{\max}(\mathbf{M}+\mathbf{N})\le\lambda_{\max}(\mathbf{M})+\lambda_{\max}(\mathbf{N})$, since by the Rayleigh quotient $\max_{\|\mathbf{v}\|=1}\mathbf{v}^\top(\mathbf{M}+\mathbf{N})\mathbf{v}\le\max_{\|\mathbf{v}\|=1}\mathbf{v}^\top\mathbf{M}\mathbf{v}+\max_{\|\mathbf{v}\|=1}\mathbf{v}^\top\mathbf{N}\mathbf{v}$.
2. **Kronecker product:** the eigenvalues of $\mathbf{M}\otimes\mathbf{N}$ are the products $\lambda_j(\mathbf{M})\lambda_k(\mathbf{N})$, so if both matrices are PSD, $\lambda_{\max}(\mathbf{M}\otimes\mathbf{N}) = \lambda_{\max}(\mathbf{M})\,\lambda_{\max}(\mathbf{N})$.
3. **Shift:** $\lambda_{\max}(\mathbf{M}+\mu\mathbf{I}) = \lambda_{\max}(\mathbf{M})+\mu$.

Applying these to (1), together with (j-1) and the fact that $\mathbf{a}_i\mathbf{a}_i^\top$ and $\mathbf{S}_i$ are PSD,
$$
\lambda_{\max}\big(\nabla^2 f_\mu(\mathbf{X})\big)\le\sum_{i=1}^n\lambda_{\max}(\mathbf{a}_i\mathbf{a}_i^\top)\,\lambda_{\max}(\mathbf{S}_i)+\mu = \sum_{i=1}^n\|\mathbf{a}_i\|_2^2\,\lambda_{\max}(\mathbf{S}_i)+\mu.
$$
We still need to bound $\lambda_{\max}(\mathbf{S}_i)$. We may use $\|\mathbf{S}_i\|_2\le 1/2$, but the looser bound $1$ is easy to see from the variance interpretation in (b): for any $\mathbf{z}$, $\mathbf{z}^\top\mathbf{S}_i\mathbf{z} = \operatorname{Var}(Z)\le\mathbb{E}[Z^2] = \sum_k p_{i,k}z_k^2\le\|\mathbf{z}\|_2^2$, since $p_{i,k}\le 1$. With $\lambda_{\max}(\mathbf{S}_i)\le 1$ we obtain, for every $\mathbf{X}$,
$$
\lambda_{\max}\big(\nabla^2 f_\mu(\mathbf{X})\big)\le\sum_{i=1}^n\|\mathbf{a}_i\|_2^2+\mu.
$$

__(j-3)__ Conclude that $f_\mu$ is $L$-smooth for $L = \|\mathbf{A}\|_F^2 + \mu$.


*Write your answer here.*

The idea is that the Hessian is the derivative of the gradient, so if the Hessian is bounded by $L$ everywhere, the gradient cannot change faster than $L$.

The rows of $\mathbf{A}$ are the $\mathbf{a}_i^\top$, so summing the squared entries row by row gives
$$
\|\mathbf{A}\|_F^2 = \sum_{i=1}^n\sum_{j=1}^p a_{ij}^2 = \sum_{i=1}^n\|\mathbf{a}_i\|_2^2 .
$$
Hence the bound from (j-2) reads $\lambda_{\max}\big(\nabla^2 f_\mu(\mathbf{X})\big)\le\|\mathbf{A}\|_F^2+\mu = L$ for every $\mathbf{X}$.

For a symmetric matrix, the spectral norm is its largest eigenvalue in absolute value. By (h) and (j-2), all eigenvalues of $\nabla^2 f_\mu(\mathbf{X})$ lie in $[\mu, L]$, so they are all positive and
$$
\|\nabla^2 f_\mu(\mathbf{X})\|_2 = \lambda_{\max}\big(\nabla^2 f_\mu(\mathbf{X})\big)\le L \qquad \text{for all } \mathbf{X}.
$$
In words, the Hessian never stretches a vector by more than a factor $L$: $\|\nabla^2 f_\mu(\mathbf{X})\,\mathbf{v}\|_2\le L\|\mathbf{v}\|_2$ for every $\mathbf{v}$.

We take two points and work with their row-wise vectorizations $\mathbf{x},\mathbf{y}\in\mathbb{R}^{pC}$, so that the gradient is a vector and the Hessian is the $pC\times pC$ matrix from (1). We move along the segment between them, $\mathbf{z}(t) = \mathbf{x}+t(\mathbf{y}-\mathbf{x})$ for $t\in[0,1]$. By the chain rule, the gradient changes along this path at rate
$$
\frac{d}{dt}\nabla f_\mu\big(\mathbf{z}(t)\big) = \nabla^2 f_\mu\big(\mathbf{z}(t)\big)\,(\mathbf{y}-\mathbf{x}),
$$
so integrating from $t=0$ to $t=1$ gives
$$
\nabla f_\mu(\mathbf{y})-\nabla f_\mu(\mathbf{x}) = \int_0^1\nabla^2 f_\mu\big(\mathbf{z}(t)\big)\,(\mathbf{y}-\mathbf{x})\,dt .
$$
Now we take norms, move the norm inside the integral, and use Step 2 at every point of the path:
$$
\|\nabla f_\mu(\mathbf{y})-\nabla f_\mu(\mathbf{x})\|_2 \le \int_0^1\underbrace{\big\|\nabla^2 f_\mu\big(\mathbf{z}(t)\big)\big\|_2}_{\le\, L}\,\|\mathbf{y}-\mathbf{x}\|_2\,dt \le L\,\|\mathbf{y}-\mathbf{x}\|_2 .
$$

Vectorizing only reorders the entries, so the Euclidean norm of a vectorized matrix equals its Frobenius norm. Writing the last inequality in matrix form, we get
$$
\|\nabla f_\mu(\mathbf{Y})-\nabla f_\mu(\mathbf{X})\|_F \le \big(\|\mathbf{A}\|_F^2+\mu\big)\,\|\mathbf{Y}-\mathbf{X}\|_F \qquad \forall\,\mathbf{X},\mathbf{Y}\in\mathbb{R}^{p\times C},
$$
which is exactly the statement that $f_\mu$ is $L$-smooth with $L=\|\mathbf{A}\|_F^2+\mu$.

__(l)__ (3 points) KL divergence and NLL. Let $q_i=(q(c\mid\mathbf{a}_i))_{c=1}^C$ be the true conditional label distribution and $\mathbf{p}_i$ the model softmax distribution. Write $\mathrm{KL}(q_i\,\|\,\mathbf{p}_i)$ and show that minimizing its average over the samples is equivalent to minimizing the expected average negative log-likelihood, where the expectation is over labels drawn from $q$ with the features held fixed. Explain how $f(\mathbf{X})/n$ from (a) estimates this expected loss.

*Write your answer here.*

By definition, the KL divergence between the true conditional distribution and the model's one is
$$
\mathrm{KL}(q_i\,\|\,\mathbf{p}_i) = \sum_{c=1}^C q(c\mid\mathbf{a}_i)\log\frac{q(c\mid\mathbf{a}_i)}{p_{i,c}} = \underbrace{\sum_{c=1}^C q(c\mid\mathbf{a}_i)\log q(c\mid\mathbf{a}_i)}_{=\,-H(q_i)} + \underbrace{\sum_{c=1}^C q(c\mid\mathbf{a}_i)\big(-\log p_{i,c}\big)}_{\text{cross-entropy } H(q_i,\,\mathbf{p}_i)},
$$
where $\mathbf{p}_i=\mathbf{p}_i(\mathbf{X})$ depends on $\mathbf{X}$ but the entropy $H(q_i)$ does not. What is important to observe is that the cross-entropy is exactly an expected negative log-likelihood: if $b_i\sim q(\cdot\mid\mathbf{a}_i)$ with $\mathbf{a}_i$ fixed, then $\mathbb{E}_{b_i\sim q_i}[-\log p_{i,b_i}] = \sum_c q(c\mid\mathbf{a}_i)(-\log p_{i,c})$. If we average over the samples,
$$
\frac1n\sum_{i=1}^n\mathrm{KL}\big(q_i\,\|\,\mathbf{p}_i(\mathbf{X})\big) = \underbrace{-\frac1n\sum_{i=1}^n H(q_i)}_{\text{independent of }\mathbf{X}} + \mathbb{E}_{b_i\sim q_i}\Big[\frac1n\sum_{i=1}^n-\log\mathbb{P}_{\mathbf{X}}(b_i\mid\mathbf{a}_i)\Big].
$$
The two objectives are the same except for a constant, so they have the same minimizers, and so minimizing the average KL to the true distribution is equivalent to minimizing the expected average NLL. By (a), this expected loss is simply $\mathbb{E}_{b\sim q}\big[f(\mathbf{X})/n\big]$, with $f$ evaluated at random labels.

This is also how we can explain the $f(\mathbf{X})/n$ question. In practice we do not know $q$, so we cannot compute the expected loss directly. What we do have is one observed label $b_i$ for each sample, drawn from $q(\cdot\mid\mathbf{a}_i)$. If we use this single label in place of the whole distribution $q_i$, so if we replace $q_i$ by the one-hot vector $\mathbf{e}_{b_i}$, the cross-entropy of sample $i$ becomes just $-\log p_{i,b_i}(\mathbf{X})$, and averaging over the samples gives
$$
\frac1n\sum_{i=1}^n H\big(\mathbf{e}_{b_i},\mathbf{p}_i(\mathbf{X})\big) = \frac1n\sum_{i=1}^n-\log p_{i,b_i}(\mathbf{X}) = \frac{f(\mathbf{X})}{n}.
$$
So $f(\mathbf{X})/n$ is the expected loss computed with the observed labels instead of the unknown $q$. For a fixed $\mathbf{X}$, the expectation of each term $-\log p_{i,b_i}(\mathbf{X})$ over $b_i\sim q_i$ is exactly the cross-entropy $H(q_i,\mathbf{p}_i)$, so on average $f(\mathbf{X})/n$ gives the right value, so it is an unbiased estimator of the expected average NLL. Since the labels are drawn independently and each term is bounded for a fixed $\mathbf{X}$, the variance of this average decreases like $1/n$, and by the law of large numbers $f(\mathbf{X})/n$ gets closer to the expected loss as $n$ grows. This means that minimizing $f(\mathbf{X})/n$, which is exactly maximum likelihood, is a practical way of finding the model that is closest in KL divergence to the true conditional distribution.

The unregularized maximum likelihood problem may have no finite minimizer. Adding a quadratic regularizer with $\mu>0$ gives a unique regularized estimate, defined by the smooth, strongly convex problem:
<a id="eq-regularized-estimator"></a>

$$
 \mathbf{X}^\star = \arg\min_{\mathbf{X} \in \mathbb{R}^{p\times C}} f(\mathbf{X}) + \frac{\mu}{2}\|\mathbf{X}\|_F^2.\tag{2}
$$


## Binary logistic regression (specialization for Part 2)

While this part analyzed the multiclass (softmax) setting, in the next exercise we will continue under the simplified two-class case.

Let labels be $b_i \in \{-1, +1\}$, features $\mathbf{a}_i \in \mathbb{R}^p$, and weight vector $\mathbf{x} \in \mathbb{R}^p$. Define the sigmoid
$$
\sigma(t) = \frac{1}{1+e^{-t}}.
$$
Model the conditional distribution as
$$
\mathbb{P}(b_i = j \mid \mathbf{a}_i) = \sigma\big(j\, \mathbf{a}_i^\top \mathbf{x}\big), \quad j \in \{-1,+1\}.
$$
The likelihood over i.i.d. samples $\{(\mathbf{a}_i, b_i)\}_{i=1}^n$ is
$$
\mathcal{L}(\mathbf{x}) = \prod_{i=1}^n \sigma\big(b_i\, \mathbf{a}_i^\top \mathbf{x}\big),
$$
so the negative log-likelihood is
$$
 f(\mathbf{x}) = -\log \mathcal{L}(\mathbf{x}) = \sum_{i=1}^n \log\big(1 + e^{-b_i\, \mathbf{a}_i^\top \mathbf{x}}\big).
$$

__(m)__ (2 points) Show that the gradient of the negative log-likelihood is the standard binary logistic regression gradient:
$$
\nabla f(\mathbf{x}) = \sum_{i=1}^n \big(-b_i\, \sigma(-b_i\, \mathbf{a}_i^\top \mathbf{x})\big)\, \mathbf{a}_i.
$$
(Hint: use the chain rule and $\sigma'(t) = \sigma(t)\big(1-\sigma(t)\big)$.)

We will use this binary formulation in Part 2 - First order methods.


*Write your answer here.*

The trick is to notice that each term of $f$ is just $-\log\sigma$ evaluated at the margin $t_i = b_i\mathbf{a}_i^\top\mathbf{x}$, since $\log(1+e^{-t}) = -\log\frac{1}{1+e^{-t}} = -\log\sigma(t)$. So $f(\mathbf{x}) = \sum_{i=1}^n\ell(b_i\mathbf{a}_i^\top\mathbf{x})$ with $\ell(t) = -\log\sigma(t)$.

Differentiating the scalar function with the given identity for $\sigma'$,
$$
\ell'(t) = -\frac{\sigma'(t)}{\sigma(t)} = -\frac{\sigma(t)\big(1-\sigma(t)\big)}{\sigma(t)} = -\big(1-\sigma(t)\big) = -\sigma(-t),
$$
where the last step uses $1-\sigma(t) = \frac{e^{-t}}{1+e^{-t}} = \frac{1}{e^{t}+1} = \sigma(-t)$.

The inner map $\mathbf{x}\mapsto b_i\mathbf{a}_i^\top\mathbf{x}$ is linear with gradient $b_i\mathbf{a}_i$, so by the chain rule $\nabla_{\mathbf{x}}\,\ell(b_i\mathbf{a}_i^\top\mathbf{x}) = \ell'(b_i\mathbf{a}_i^\top\mathbf{x})\,b_i\mathbf{a}_i = -\sigma(-b_i\mathbf{a}_i^\top\mathbf{x})\,b_i\mathbf{a}_i$. Summing over the samples,
$$
\nabla f(\mathbf{x}) = \sum_{i=1}^n\big(-b_i\,\sigma(-b_i\,\mathbf{a}_i^\top\mathbf{x})\big)\,\mathbf{a}_i.
$$
As an interpretation, $\sigma(-b_i\mathbf{a}_i^\top\mathbf{x}) = 1-\mathbb{P}(b_i\mid\mathbf{a}_i)$ is the probability the model assigns to the *wrong* label, so confidently well-classified points barely contribute to the gradient. This mirrors the error term $\mathbf{p}_i-\mathbf{e}_{b_i}$ of the multiclass gradient in (f).